## Example notebook #1

This notebook shows the basic functionality of the pseudospectral time-splitting algorithms

In [1]:
# Array module
import numpy as np

from numpy.polynomial.hermite import hermgauss # Nodes and weights for Hermite-Gauss quadrature
from numpy import exp, abs

# Graphics module
import matplotlib.pyplot as plt   # 2D figures
from mpl_toolkits import mplot3d  # 3D representations

# Embed interactive widgets in Jupyter notebook
%matplotlib widget 


The module *base_functions* contains definitions for variuos orthogonal bases (i.e. the Hermite functions used in this example)

In [2]:
from base_functions import hermite_function

In the next cell, we define the symbol for the pseudodifferential operator. Essentially, it represents the dispersion relation for the linear part of the evolution operator. Currently, the implementation is a bit messy but it is useful for testing purposes. Using "schrodinger" we get the Nonlinear Schrödinger equation (NLSE).

In [3]:
B_TYPE = 'schrodinger' # 'gaussian'
B_ALTO = 2.0
B_ANCHO = 2.


#Función que define el operador pseudo-diferencial

def b_scalar(k, k0=0., alt = B_ALTO, ancho = B_ANCHO):

    if abs(k) < ancho:
        return 1.
    
    return 0. # 1. / ancho * abs(k)

if (B_TYPE == 'interval'):
    b = np.vectorize(b_scalar)
elif (B_TYPE == 'gaussian'):
    def b(k):
        return hermite_function(0, 0.5*k)/np.sqrt(2)
        #return(1 / (1 + k**4))
elif (B_TYPE == 'inv-gaussian'):
    def b(k):
        return 1 - np.exp(-0.2 * k ** 2)   #hermite_function(0,0) - hermite_function(0, 0.5 * k)
        #return(1 - 1 / (1 + 0.5*k**2))
elif (B_TYPE == 'disp_gaussian'):
    def b(k):
        return hermite_function(0, k - B_ALTO)
elif (B_TYPE == 'sym_gaussian'):
    def b(k):
        return((hermite_function(0, k + B_ALTO) + hermite_function(0, k - B_ALTO)))

elif (B_TYPE == 'unsym_gaussian'):
    def b(k):
        return (hermite_function(0, k + B_ALTO) + 2 * hermite_function(0, k - B_ALTO))
elif (B_TYPE == 'schrodinger'):
    def b(k):
        return (0.5 * k**2)
elif (B_TYPE == 'absolute'):
    def b(k):
        return abs(k)*0.1

Initialize the linear and nonlinear propagators for the time-splitting scheme.
Each propagator expects a state in the correct representation (i.e. function values at the collocation points). The initialization functions are found in the _pseudospectral_ module and must be tailored to the specific operators used.

In [4]:
from pseudospectral import P_A_init, P_B_init

alpha = -3.710
sigma = 2.
dim_base = 150

P_A = P_A_init(alpha=alpha, sigma=sigma) # Create nonlinear propagator
P_B = P_B_init(b, dim_base=dim_base)     # Create linear propagator

x, w = hermgauss(dim_base)               # Nodes and weights for Hermite-Gauss quadrature

Define the initial state $u_0$ (currently it is a soliton solution for the NLSE)

In [5]:
delta = 0.0
u0 = 1. / np.sqrt(2.) / np.cosh(x) * exp(-2*np.pi*1.0j*x*delta) # (1. / np.sqrt(2.) / np.cosh(x-10) * exp(-2*np.pi*1.0j*x*delta) + 1. / np.sqrt(2.) / np.cosh(x+10) * exp(2*np.pi*1.0j*x*delta))
# u0 = hermite_function(0, x)* exp(-2*np.pi*1.0j*x*delta) 

Plot the modulus of the initial state $u_0$

In [6]:
plt.figure()
plt.plot(x, np.abs(u0))

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Do the evolution using the function _evolve_ from the _splitting_ module

In [7]:
from splitting import evolve

T = 50.      # Total evolution time
dt = 0.001   # Time step for the splitting algorithm

t, sol = evolve(u0, dt, T, P_A, P_B)      

The solution is stored in a 2D matrix _sol_, which has shape

In [8]:
sol.shape

(50000, 150)

Show the modulus squared of the solution

In [9]:
plt.figure()
plt.imshow(abs(sol[::500,:]**2), cmap='viridis')
plt.colorbar()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

From the solution, calculate some important quantities: $L_p$ norms, center of mass, variance and energy

In [10]:
from quantities import norm, center_of_mass, variance, kinetic_energy

L2 = norm(sol, w*np.exp(x**2), 2)
Lsigma = norm(sol, w*np.exp(x**2), 2 * sigma + 2)
var = variance(sol**2, w*np.exp(x**2), x)
CM = center_of_mass(sol**2, w*np.exp(x**2), x)
KE = kinetic_energy(sol, b)
energy = 0.5 * KE + 1 / (2 * sigma + 2) * alpha * Lsigma ** (2 * sigma + 2)

/home/lisandro/Nextcloud/Doctorado/Software/pseudosplit/quantities.py:86: ComplexWarning: Casting complex values to real discards the imaginary part
  KE[i] = beta[i,:].dot(C.dot(beta[i,:].conj()))


It can be seen from the following figure that the algorithm preserves the $L_2$ norm ("mass") and hamiltonian ("energy"), which are conserved quantities.

In [39]:
plt.figure()

a1 = plt.subplot2grid((1,2), (0,0))
a2 = plt.subplot2grid((1,2), (0,1))

a1.set_ylim(-1.2*np.max(energy), 1.2*np.max(energy) )
a2.set_ylim(0., 1.2*np.max(L2))

a1.set_title("Energy")
a2.set_title("$L_2$")

a1.plot(t, energy)
a2.plot(t, L2)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [40]:
plt.figure()
plt.plot(x, np.abs(sol[49000,:])**2, x, u0**2)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

/home/lisandro/anaconda3/lib/python3.7/site-packages/numpy/core/_asarray.py:83: ComplexWarning: Casting complex values to real discards the imaginary part
  return array(a, dtype, copy=False, order=order)


Show the modulus squared of the solution $u(x,t)$ in two ways

In [44]:
fig = plt.figure()

ax1a = fig.add_subplot(121, projection='3d')
ax1a.set_title('Evolution of $|u|^2$')
ax1a.set_xlabel('$t$')
ax1a.set_ylabel('$x$')
ax1a.set_zlabel('$|u(x,t)|^2$')

tg, xg = np.meshgrid(t, x, indexing='ij')
ax1a.plot_wireframe(tg,xg,np.abs(sol)**2, cstride=0, rstride=5000, colors=((1,0,1), (0,0.5,1), (0,0,1)))

ax1b = fig.add_subplot(122)
ax1b.imshow(np.abs(sol[::500])**2)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Show various quantities

In [14]:
plt.figure()
ax2 = plt.subplot2grid((2,2), (0,0))
ax3 = plt.subplot2grid((2,2), (0,1))
ax4 = plt.subplot2grid((2,2), (1,0))
ax5 = plt.subplot2grid((2,2), (1,1))


ax2.set_title('Energy')
ax2.set_xlim(0,T)
# ax2.set_xlabel('t')
ax2.set_ylim(-1.2 * np.max(energy), 1.2 * np.max(energy)) # * np.max(alpha / 4.0 * L4**4 + 0.5 * energy1))
ax2.plot(t,  energy ) 

ax3.set_title('Variance')
ax3.set_xlim(0,T)
ax3.set_ylim(0., 1.2 * np.max(var))
ax3.plot(t, var)

#ax4.set_title('Localización')
ax4.set_title('$L_2$')
ax4.set_xlim(0,T)
#ax4.set_ylim(0,1.1*np.max(L2_loc))
ax4.set_ylim(0,1.1*np.max(L2))

#ax4.plot(t, L2_loc)
ax4.plot(t, L2)

ax5.set_xlim(0,T)

ax5.set_title('Centre of mass')
ax5.set_ylim(-1.2*np.max(abs(CM)), 1.2*np.max(abs(CM)) )
ax5.plot(t, CM)

#fig.tight_layout()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

The function *hermite_projection_matrix* returns the matrix that converts between real space representation (function values at collocation points) and Hermite representation (coefficients for the Hermite function series)

In [15]:
from pseudospectral import hermite_projection_matrix
U = hermite_projection_matrix(dim_base)

In [31]:
plt.figure()
plt.imshow(abs(U), cmap='viridis')
plt.colorbar()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Calculate the coefficients for the Hermite and Fourier expansions from solution

In [17]:
from quantities import hermite_coeffs, fourier_coeffs

In [45]:
h_coeffs = hermite_coeffs(sol)
f_coeffs = fourier_coeffs(sol, numpoints=150)

In [46]:
h_coeffs.shape

(50000, 150)

In [47]:
f_coeffs.shape

(50000, 150)

Show the evolution of Fourier coefficients (modulus, phase)

In [58]:
fig = plt.figure()

ax1 = fig.add_subplot(131, projection='3d')
tg, xg = np.meshgrid(t, x, indexing='ij')
ax1.plot_wireframe(tg,xg,np.abs(f_coeffs), cstride=0, rstride=5000, colors=((1,0,1), (0,0.5,1), (0,0,1)))

ax2 = fig.add_subplot(132)
ax2.imshow(np.abs(f_coeffs[::500,:]), cmap='viridis')

ax2 = fig.add_subplot(133)
ax2.imshow(np.angle(f_coeffs[::500,:]), cmap='viridis')


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Show the evolution of Hermite coefficients

In [60]:
fig = plt.figure()

ax1 = fig.add_subplot(121, projection='3d')
tg, xg = np.meshgrid(t, x, indexing='ij')
ax1.plot_wireframe(tg,xg,np.abs(h_coeffs), cstride=0, rstride=1000, colors=((1,0,1), (0,0.5,1), (0,0,1)))

ax2 = fig.add_subplot(122)

ax2.imshow(np.abs(h_coeffs[::500,:]))

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …